# Last.fm Data Cleaning

This notebook standardizes artist and track names, removes duplicates and extracts additional metadata used in later stages of the analysis.

## 1. Load libraries

In [ ]:
import pandas as pd
from pathlib import Path
import re

## 2. Load merged dataset

Load the merged Last.fm listening history produced in the previous notebook.

In [ ]:
interim_df = pd.read_parquet("../data/interim/lastfm/lastfm_scrobbles_merged.parquet")

## 3. Initial dataset inspection

Perform a quick quality check before cleaning.

In [ ]:
interim_df.shape

In [ ]:
interim_df.head()

In [ ]:
interim_df["artist"].value_counts().head(20)

## 4. Timestamp processing

Remove incomplete records and convert Unix timestamps into datetime features used in the temporal analysis.

In [ ]:
df = interim_df.dropna(subset=["timestamp"])
df

In [ ]:
# Convert timestamps to datetime and extract year and month
df["timestamp"] = pd.to_numeric(df["timestamp"], errors="coerce")
df["timestamp"] = df["timestamp"].astype(int)
df["played_at"] = pd.to_datetime(df["timestamp"], unit="s")
df["year"] = df["played_at"].dt.year
df["month"] = df["played_at"].dt.to_period("M")
df

In [ ]:
print(f"{df.shape} records after processing.")
print(f"Number of empty rows {df.isna().sum()}")
print(f"Number of unique artists in the df {df.artist.nunique()}")
print(f"Number of unique tracks in the df {df.track.nunique()}")
print(f"Top rows {df.head()}")

## 5. Remove duplicate records

Remove exact duplicate scrobbles based on artist, track and timestamp.

In [ ]:
df.duplicated(subset=["artist", "track", "timestamp"]).sum()
df = df.drop_duplicates(subset=["artist", "track", "timestamp"])
df

## 6. Artist name normalization

Standardize artist names to improve consistency across the dataset.

In [ ]:

def clean_text_artist(text):
    if text is None:
        return None
    
    text = text.lower()
    
    # remove brackets (artist rarely has any important info in brackets)
    text = re.sub(r"\(.*?\)", "", text)
    
    # remove strange suffixes
    text = re.sub(r"\d{2,}[a-z]{3,}\d*$", "", text)
    
    # remove " - something"
    text = re.sub(r"\s-\s.*", "", text)

    # remove special characters
    text = re.sub(r"[^a-z0-9\s]", "", text)

    # normalize whitespace
    text = re.sub(r"\s+", " ", text).strip()

    return text if text else None

## 7. Track title normalization

Remove edition labels and other non-essential information from track titles while preserving the original song identity.

In [ ]:
def clean_text_track(text):
    if text is None:
        return None
    
    text = text.lower()
    
    # Remove content in parentheses that is not part of the title
    text = re.sub(
        r"\((?:"
        r"feat\.?.*?|ft\.?.*?|"
        r"remaster(?:ed)?(?: \d{4})?|"
        r"live(?: at .*?)?|"
        r"radio edit|edit|"
        r"(?:acoustic|alternative|instrumental)(?: version)?|"
        r"mix|version|"
        r"deluxe|bonus track|"
        r"single version"
        r")\)",
        "",
        text,
    )
    
    # remove feat/ft also if outside brackets
    text = re.sub(r"\b(feat|ft)\.?\b.*", "", text)
    
    # 🔥 remove remix/rework/edit/version also if outside brackets
    text = re.sub(r"\b(remix|rework|edit|version)\b.*", "", text)
    
    # remove strange suffixes
    text = re.sub(r"\d{2,}[a-z]{3,}\d*$", "", text)
    
    # remove " - something"
    text = re.sub(r"\s-\s.*", "", text)

    # remove special characters
    text = re.sub(r"[^a-z0-9\s]", "", text)

    # normalize whitespace
    text = re.sub(r"\s+", " ", text).strip()

    return text if text else None

In [ ]:
df["artist_clean"] = df["artist"].apply(clean_text_artist)
df["track_clean"] = df["track"].apply(clean_text_track)

In [ ]:
df = df.dropna(subset=["track_clean"])

## 8. Cleaning validation

Compare the number of unique artists and tracks before and after normalization.

In [ ]:
print("Before:", df["track"].nunique())
print("After:", df["track_clean"].nunique())

print("Before:", df["artist"].nunique())
print("After:", df["artist_clean"].nunique())

## 9. Parenthetical content analysis

Inspect the most common parenthetical expressions in track titles to guide further cleaning rules.

In [ ]:
df[["track", "track_clean"]].sample(50)

In [ ]:
df[["artist", "artist_clean"]].sample(50)

In [ ]:

df["parentheses"] = df["track"].apply(lambda x: re.findall(r"\(.*?\)", x.lower()))

In [ ]:
df["parentheses"].explode().value_counts().head(30)

## 10. Export cleaned dataset

Save the cleaned dataset for the genre-tag enrichment stage.

In [ ]:
df.shape

In [ ]:
df.to_parquet(
    "../data/processed/lastfm_scrobbles_clean.parquet",
    index=False
)

### Output

`data/processed/lastfm_scrobbles_clean.parquet`

Used in **03_lastfm_tags_enrichment.ipynb**.